In [1]:
import os
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from datetime import datetime
import sys
sys.path.append(os.path.abspath("../../"))
from utils.soa_helpers import prepare_global_data, evaluate_model_global, generate_shap_summary, feature_selection

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = os.getenv("ROOT_DIR")

/Volumes/T7/documents/github/dropout-prediction/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TRAIN_PATH = os.path.join(ROOT_DIR, ".data/train_test/train_timeseries.csv")
TEST_PATH = os.path.join(ROOT_DIR, ".data/train_test/test_timeseries.csv")
OUTPUT_DIR = os.path.join(ROOT_DIR, "outputs/runs_soa")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")

for path in [OUTPUT_DIR, MODELS_DIR, PLOTS_DIR]:
    os.makedirs(path, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
LAGS = [7, 14]
USE_FS = True
USE_MACRO = False
TIME_AGG = 'flatten'

CLASSIFIERS = {
    'LogisticRegression': {
        'estimator': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
        'params': {
            'clf__penalty': ['l2', 'l1'],
            'clf__C': [0.01, 0.1, 1.0, 10.0], 
            'clf__solver': ['liblinear']}
    },
    'RandomForest': {
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
        'params': {
            'scaler': ['passthrough'],
            'clf__n_estimators': [200, 500], 
            'clf__max_depth': [10, 15, None], 
            'clf__min_samples_leaf': [1, 2],
            'clf__max_features': ['sqrt', 'log2']}
    }
}

In [4]:
results_list = []

for current_lag in LAGS:
    print(f"[INFO] Pipeline for lag {current_lag} days")

    print("[INFO] Processing train set...")
    X_train, y_train, y_strat_train = prepare_global_data(
        TRAIN_PATH, lag=current_lag, use_macro=USE_MACRO, time_aggregation=TIME_AGG
    )
    
    print("[INFO] Processing test set...")
    X_test, y_test, _ = prepare_global_data(
        TEST_PATH, lag=current_lag, use_macro=USE_MACRO, time_aggregation=TIME_AGG
    )

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    if USE_FS and not USE_MACRO:
        X_train, X_test, selected_features = feature_selection(
            X_train, y_train, X_test, corr_threshold=0.75
        )

    print(f"[INFO] Final shape: \nTrain {X_train.shape} | Test: {X_test.shape}")

    for algo_name, model_config in CLASSIFIERS.items():
        print(f"[INFO] Training Model: {algo_name}...")
        
        metrics, best_model = evaluate_model_global(
            model_config, X_train, y_train, y_strat_train, X_test, y_test
        )
        
        print(f"[INFO] Results: \nPR-AUC: {metrics['pr_auc']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}")
        
        model_name = f"{algo_name}_fs{USE_FS}"
        model_path = os.path.join(MODELS_DIR, f"{model_name}_lag{current_lag}.pkl")
        joblib.dump(best_model, model_path)
        print(f"[INFO] Model saved in {model_path}")
        
        generate_shap_summary(best_model, X_train, X_test, model_name, current_lag, PLOTS_DIR)

        result_row = {
            'lag': current_lag,
            'algorithm': algo_name,
            'use_macro': USE_MACRO,
            'time_agg': TIME_AGG,
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'roc_auc': metrics['roc_auc'],
            'pr_auc': metrics['pr_auc']
        }
        results_list.append(result_row)

if results_list:
    df_results = pd.DataFrame(results_list)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_csv = os.path.join(OUTPUT_DIR, f"soa_metrics_{TIME_AGG}_{timestamp}.csv")
    df_results.to_csv(output_csv, index=False)
    print(f"\n[INFO] Succeeded, saved in: {output_csv}")

[INFO] Pipeline for lag 7 days
[INFO] Processing train set...
[INFO] Processing test set...
[INFO] Starting Feature Selection...


AssertionError: [CRITICAL] len['abandoned_day_1', 'abandoned_day_2', 'abandoned_day_3', 'abandoned_day_4', 'abandoned_day_5', 'abandoned_day_6', 'abandoned_day_7', 'accepted_day_1', 'accepted_day_2', 'accepted_day_3', 'accepted_day_4', 'accepted_day_5', 'accepted_day_6', 'accepted_day_7', 'add_day_1', 'add_day_2', 'add_day_3', 'add_day_4', 'add_day_5', 'add_day_6', 'add_day_7', 'add contact_day_1', 'add contact_day_2', 'add contact_day_3', 'add contact_day_4', 'add contact_day_5', 'add contact_day_6', 'add contact_day_7', 'add discussion_day_1', 'add discussion_day_3', 'add discussion_day_4', 'add discussion_day_5', 'add entry_day_1', 'add entry_day_2', 'add entry_day_3', 'add entry_day_4', 'add entry_day_5', 'add entry_day_6', 'add entry_day_7', 'add page_day_1', 'add page_day_2', 'add page_day_3', 'add page_day_4', 'add page_day_5', 'add page_day_6', 'add page_day_7', 'add post_day_1', 'add post_day_2', 'add post_day_3', 'add post_day_4', 'add post_day_5', 'add post_day_6', 'add post